In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 3.2 MB/s  0:00:01 eta 0:00:01


In [7]:
from Bio import Entrez
import pandas as pd
import chromadb
import os
from dotenv import load_dotenv
from openai import OpenAI

In [4]:
Entrez.email = 'mateuszczeresnia@gmail.com'

search_handle = Entrez.esearch(
    db = 'pubmed',
    term = 'microRNA AND ischemic stroke',
    retmax = 20,
    sort = 'relevance'
)
search_results = Entrez.read(search_handle)
search_handle.close()

pmid_list = search_results['IdList']
print(f'Znaleziono {len(pmid_list)} artykułów. Przykładowe PMID:', pmid_list[:20])

Znaleziono 20 artykułów. Przykładowe PMID: ['29948938', '38928006', '35189704', '39468286', '27691978', '36657721', '39859461', '40981170', '34244338', '37787681', '36967397', '33368054', '36596006', '36052307', '37603533', '35421904', '40526299', '39538152', '36469178', '36244267']


In [5]:
fetch_handle = Entrez.efetch(
    db = 'pubmed',
    id = ','.join(pmid_list),
    rettype = 'medline',
    retmode = 'xml'
)
records = Entrez.read(fetch_handle)
fetch_handle.close()

In [22]:
publications = []
for article in records['PubmedArticle']:
    medline = article['MedlineCitation']
    pmid = str(medline['PMID'])
    art_data = medline ['Article']
    title = str(art_data.get('ArticleTitle', ''))

    if 'Abstract' in art_data and 'AbstractText' in art_data['Abstract']:
        abstract_parts = [str(part) for part in art_data ['Abstract']['AbstractText']]
        full_abstract = ' '.join(abstract_parts)

        pub_date = art_data['Journal']['JournalIssue']['PubDate']
        year = int(pub_date.get('Year', 2024))

        publications.append({
            'pmid': f'PMID_{pmid}',
            'title': title,
            'abstract': full_abstract,
            'year': year
        })

df_pubmed = pd.DataFrame(publications)
print(f'Pobrano {len(df_pubmed)} pełnych abstraktów')
df_pubmed

Pobrano 20 pełnych abstraktów


,pmid,title,abstract,year
0,PMID_29948938,"Monogenic, Polygenic, and MicroRNA Markers for...",Ischemic stroke (IS) is a leading disease with...,2019
1,PMID_38928006,Molecular Pathogenesis of Ischemic and Hemorrh...,Stroke represents one of the neurological dise...,2024
2,PMID_35189704,CircOGDH Is a Penumbra Biomarker and Therapeut...,Acute ischemic stroke (AIS) is a leading cause...,2022
3,PMID_39468286,The foam cell-derived exosomal miRNA Novel-3 d...,Large artery atherosclerosis (LAA) is a preval...,2024
4,PMID_27691978,The Role of microRNA in Ischemic and Hemorrhag...,Stroke is the second leading cause of death wo...,2017
5,PMID_36657721,microRNA as a therapeutic for ischemic stroke.,microRNA (miRNA) are important regulators of g...,2023
6,PMID_39859461,MicroRNA Expression Profile in Acute Ischemic ...,Acute ischemic stroke with large vessel occlus...,2025
7,PMID_40981170,The Association of MicroRNA-21 with Carotid Ar...,Ischemic stroke is one of the leading causes o...,2025
8,PMID_34244338,Analysis of microRNA signatures in ischemic st...,We investigated the microRNA expression patter...,2022
9,PMID_37787681,Prognostic utility of MicroRNA-221 and interle...,Cerebral ischemic stroke has a significant mor...,2023


In [23]:
def chunking(text, piece_size = 200, bookmark = 50):
    piece = []
    step = piece_size - bookmark

    for i in range(0, len(text), step):
        clip = text[i : i + piece_size]
        piece.append(clip)
    return piece

In [24]:
client = chromadb.PersistentClient(path = '/Users/mateurz/VS Code Workspace Folder/Project')
pubmed_collection = client.get_or_create_collection(name = 'pubmed_stroke_rag')

ALL_ids = []
ALL_documents = []
ALL_metadates = []

for _, row in df_pubmed.iterrows():
    pieces = chunking(row['abstract'], piece_size = 600, bookmark = 100)

    for idx, piece in enumerate(pieces):
        ALL_ids.append(f'{row['pmid']}_chunk_{idx}')
        ALL_documents.append(piece)
        ALL_metadates.append({
            'pmid': row['pmid'],
            'title': row['title'],
            'year': row['year'],
            'chunk': idx
        })
pubmed_collection.add(
    ids = ALL_ids,
    documents = ALL_documents,
    metadatas = ALL_metadates
)

print('Liczba Wektorów w nowej bazie PubMed:', pubmed_collection.count())

Liczba Wektorów w nowej bazie PubMed: 70


In [25]:
!pip install streamlit

In [1]:
!pip install langchain langchain-chroma langchain-google-genai langchain-text-splitters

  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 6.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 2.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 6.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.4/640.4 kB 1.2 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: certifi
    Found existing installation: certifi 2026.7.22
    Uninstalling certifi-2026.7.22:
      Successfully uninstalled certifi-2026.7.22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23/23 [langchain]23 [langchain-google-genai]


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=20
)

przykladowy_tekst = (
    "Ischemic stroke occurs when a blood vessel supplying blood to the brain is obstructed. "
    "MicroRNAs are small non-coding RNA molecules that regulate gene expression post-transcriptionally. "
    "Recent studies show that miR-9-5p promotes angiogenesis and protects the blood-brain barrier integrity."
)

kawalki = splitter.split_text(przykladowy_tekst)
for i, k in enumerate(kawalki):
    print(f"Chunk {i}: {k}")

Chunk 0: Ischemic stroke occurs when a blood vessel
Chunk 1: when a blood vessel supplying blood to the brain
Chunk 2: blood to the brain is obstructed. MicroRNAs are
Chunk 3: MicroRNAs are small non-coding RNA molecules that
Chunk 4: RNA molecules that regulate gene expression
Chunk 5: gene expression post-transcriptionally. Recent
Chunk 6: Recent studies show that miR-9-5p promotes
Chunk 7: miR-9-5p promotes angiogenesis and protects the
Chunk 8: and protects the blood-brain barrier integrity.


In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough

# 1. Podpinamy się pod Twoją istniejącą bazę na dysku
db_client = chromadb.PersistentClient(path="/Users/mateurz/VS Code Workspace Folder/Project")
default_ef = embedding_functions.DefaultEmbeddingFunction()

vectorstore = Chroma(
    client=db_client,
    collection_name="pubmed_stroke_rag",
    embedding_function=default_ef
)

# Zamieniamy bazę w Retriever (zwracający 3 najlepsze fragmenty)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2. Funkcja pomocnicza sklejająca dokumenty i metadane w jeden tekst kontekstu
def format_docs(docs):
    return "\n\n".join(
        f"PMID: {doc.metadata.get('pmid')} | Year: {doc.metadata.get('year')} | Text: {doc.page_content}"
        for doc in docs
    )

# 3. Szablon promptu RAG
rag_prompt = ChatPromptTemplate.from_template("""
Odpowiedz po polsku na pytanie użytkownika, opierając się WYŁĄCZNIE na poniższym kontekście.
Przy każdej informacji podaj w nawiasie PMID oraz rok publikacji.
Jeśli w kontekście nie ma odpowiedzi, napisz "Nie mam takich danych w bazie".

Kontekst:
{context}

Pytanie: {question}
""")

# 4. Pełny łańcuch RAG (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

# Testujemy!
print(rag_chain.invoke("What is the role of miRNA in ischemic stroke?"))

In [11]:
llm_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

for m in llm_client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [12]:
pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 337.6 kB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 392.1 kB/s  0:00:36m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.6/138.6 MB 670.4 kB/s  0:02:330:00:0100:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sentence-transformers]ence-transformers]
Note: you may need to restart the kernel to use updated packages.


In [14]:
import chromadb

client = chromadb.PersistentClient(path="/Users/mateurz/VS Code Workspace Folder/Project/RAG_STROKE")
for col in client.list_collections():
    print(f"Kolekcja: {col.name} -> Folder UUID: {col.id} (dokumentów: {col.count()})")

Kolekcja: miRNA_test -> Folder UUID: 4f599d88-5844-4337-b42e-8157a585f359 (dokumentów: 8)
Kolekcja: pubmed_stroke_rag -> Folder UUID: be26c5bb-2ce3-4414-a90b-ab875b9c706d (dokumentów: 142)
Kolekcja: stroke_test -> Folder UUID: fd364c14-743d-44d8-a471-270a889aae99 (dokumentów: 3)
